# Prior-Art Search using RL
## Agentic Search | Multi-Turn | Tool-use  

The objective is to create an Agent to perform prior-art search over patent corpora. mainly The Harvard USPTO Patent Dataset.



In [21]:
from huggingface_hub import notebook_login
import os
import json
import chromadb
from chromadb.api.types import Embeddable, EmbeddingFunction
from chromadb.utils import embedding_functions
import asyncio


In [22]:
# Fields to extract from raw JSON data
RELEVANT_FIELDS = [
    # Identifiers & Linking
    "publication_number",
    "application_number",
    "patent_number",
    # Dates (as epoch ints)
    "date_published",
    "filing_date",
    "patent_issue_date",
    "abandon_date",
    # Status & Classes
    "decision",
    "main_cpc_label",
    "main_ipcr_label",
    # Retrievable Text
    "title",
    "abstract",
    "claims",  ## The legally enforceable boundaries of the invention — the essence of what’s protected.
    # "summary",
]

def get_IP_data():
    """Load and filter IP data from JSON files, skipping files with decode errors."""
    ip_files = []
    for file in os.listdir("Patent_data"):
        file_path = os.path.join("Patent_data", file)
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                filtered = {key: value for key, value in data.items() if key in RELEVANT_FIELDS}
                ip_files.append(filtered)
        except (UnicodeDecodeError, json.JSONDecodeError) as e:
            print(f"Skipping {file}: {e}")
    return ip_files

patent_data = get_IP_data()

Skipping .DS_Store: 'utf-8' codec can't decode byte 0x80 in position 3131: invalid start byte


#### Designing the multi-turn environment

An episode is “one prior-art search task”.


Initial observation is something like:

You are a patent prior-art search assistant.
Here is a new invention description:
<DESCRIPTION>
You have access to tools:

patent_search(query) – searches in a database of patents (title+abstract).

patent_lookup(patent_id) – shows full details of one patent.
Use the tools as needed, then produce a final ranked list of relevant prior-art patents.


The environment passes this as the initial prompt.

After each tool call, the environment appends the tool output to the conversation and gives that back to the model.

3.2. Actions

    Action = “next model output”. You constrain the format so the model can:

    Call tools:
    e.g. CALL_TOOL patent_search: "solar panel with transparent coating for windows"

    Or produce final answer:
    e.g. FINAL_ANSWER: [US1234..., EP5678..., ...] plus explanation.

The environment will:

    Parse the model’s output.

    If it’s CALL_TOOL, execute the underlying Python function and append result.

    If it’s FINAL_ANSWER, end the episode and compute reward.

This is exactly the pattern used in TRL’s experimental “learning tools” examples with calculator/wiki/python environments. 


3.3. Termination

Terminate when:

    The model outputs FINAL_ANSWER, or

    You hit max_steps (e.g. 5–8 tool calls) and then force the model to give a final answer, or mark as failure.

In [24]:
CHROMA_DB_DIR = ".chroma_db"
_chroma_semaphore: asyncio.Semaphore | None = None

def get_chroma_semaphore() -> asyncio.Semaphore:
    global _chroma_semaphore
    if _chroma_semaphore is None:
        _chroma_semaphore = asyncio.Semaphore(20)
    return _chroma_semaphore


def init_chroma_collection(force_recreate: bool = False) :
    """Connect to an existing ChromaDB collection for patent data, or create if not exists."""
    chroma_client = chromadb.PersistentClient(path=CHROMA_DB_DIR)
    # Try to get the collection if it exists
    try:
        if force_recreate:
            chroma_client.delete_collection(name="patent_collection")
        collection = chroma_client.get_collection(
            name="patent_collection",
            embedding_function=embedding_functions.SentenceTransformerEmbeddingFunction(
                model_name="sentence-transformers/all-mpnet-base-v2"
            )
        )
        # If collection exists, return it
        return collection
    except Exception:
        # If not found, create it and add data
        collection = chroma_client.get_or_create_collection(
            name="patent_collection",
            embedding_function=embedding_functions.SentenceTransformerEmbeddingFunction(
                model_name="sentence-transformers/all-mpnet-base-v2"
            ),
            configuration={
                "hnsw": {
                    "space": "cosine",
                    "ef_construction": 200,
                    "ef_search": 150
                }
            }
        )
        collection.add(
            documents=[patent["abstract"] for patent in patent_data],  # Using abstract as the main text for embeddings
            ids=[patent["publication_number"] for patent in patent_data],
            metadatas=[{k: v for k, v in patent.items() if k != 'abstract'} for patent in patent_data],
        )
        return collection

collection = init_chroma_collection(force_recreate=False)


In [25]:
collection.get("US20180137422A1-20180517")

{'ids': ['US20180137422A1-20180517'],
 'embeddings': None,
 'documents': ['Methods of training Boltzmann machines include rejection sampling to approximate a Gibbs distribution associated with layers of the Boltzmann machine. Accepted sample values obtained using a set of training vectors and a set of model values associate with a model distribution are processed to obtain gradients of an objective function so that the Boltzmann machine specification can be updated. In other examples, a Gibbs distribution is estimated or a quantum circuit is specified so at to produce eigenphases of a unitary.'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'main_cpc_label': 'G06N5022',
   'main_ipcr_label': 'G06N502',
   'title': 'FAST LOW-MEMORY METHODS FOR BAYESIAN INFERENCE, GIBBS SAMPLING AND DEEP LEARNING',
   'claims': '1.-15. (canceled) 16. A method, comprising: with a processor: obtaining a set of N samples from an initial distribution, wherein N is a 

### Defining The Tools

In [26]:
from pydantic import BaseModel, Field
from dataclasses import asdict, dataclass
from typing import List, Optional

# Patent data models
class Patent(BaseModel):
    publication_number: str
    title: str  
    abstact: Optional[str] = None
    main_ipcr_label: Optional[str] = None
    main_cpc_label: List[str] = []  
    decision: List[str] = []  
    patent_issue_date: List[str] = []  


@dataclass
class SearchResult:
    message_id: str
    snippet: str

class FinalAnswer(BaseModel):   
    answer: str
    patent_ids: list[str]


## search tool
async def search_patents(query: str, n_results: int = 10) -> list[dict]:
    """Search for top 10 relevant patents using title embedding similarity."""
    async with get_chroma_semaphore():
        results = collection.query(
            query_texts=[query],
            n_results=n_results
        )
    
    if not results:
        raise ValueError(f"No results found for query: {query}")
    if not results["metadatas"]:
        raise ValueError(f"No results metadata found for query: {query}")
    
    output = []
    for i in range(len(results["ids"][0])):
        patent_title = results["metadatas"][0][i]["title"]
        publication_number = results["ids"][0][i]
        similarity_score = results["distances"][0][i]
        output.append({"patent_title": patent_title, 
                       "publication_number": publication_number,
                       "similarity_score": similarity_score})
    
    return output

# Patent lookup tool
async def lookup_patent(publication_number: str) -> dict:
    """Lookup patent details by publication number."""
    sem = get_chroma_semaphore()
    async with sem:
        results = await asyncio.to_thread(
            collection.get,
            ids=[publication_number],
        )

    if not results or not results.get("metadatas"):
        raise ValueError(f"No patent found with publication number: {publication_number}")

    patent_content = results["documents"][0]
    patent_metadata = results["metadatas"][0]
    return {**patent_metadata, "abstract": patent_content}


def return_final_answer(
        answer: str, 
        reference_message_ids: list[str]) -> FinalAnswer:
        """Return the final answer and the message IDs of the emails that were used to generate the answer."""
        return FinalAnswer(answer=answer, source_ids=reference_message_ids)

In [27]:
await search_patents("battery management", n_results=3)

[{'patent_title': 'QUICK-EXCHANGE BATTERY ASSEMBLY, AND MOTOR VEHICLE, IN PARTICULAR MOTOR SCOOTER',
  'publication_number': 'US20180151860A1-20180531',
  'similarity_score': 0.5740572214126587},
 {'patent_title': 'METHOD FOR CALCULATING A SETPOINT FOR MANAGING THE FUEL AND ELECTRICITY CONSUMPTION OF A HYBRID MOTOR VEHICLE',
  'publication_number': 'US20180281620A1-20181004',
  'similarity_score': 0.6117575168609619},
 {'patent_title': 'POSITIVE ELECTRODE ACTIVE MATERIAL FOR RECHARGABLE LITHIUM BATTERY, METHOD FOR MANUFACTURING SAME, AND RECHARGABLE LITHIUM BATTERY INCLUDING SAME',
  'publication_number': 'US20180254473A1-20180906',
  'similarity_score': 0.6518891453742981}]

## Reward Design for prior-art Search

### Backward-citation

We need labels for what good prior art looks like.
since i dont have 'citations' in the documents
the ground truth will be basically patents abstract snippets and their backward citations as ground truth.
For a given patent P, the cited patents = “true” prior art for P.


In [ ]:
import dspy
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv()
base_url = os.getenv("GEMINI_API_BASE", "https://generativelanguage.googleapis.com/v1beta")
api_key = os.getenv("GEMINI_API_KEY")
lm = dspy.LM('gemini/gemini-2.0-flash', api_key=api_key)
dspy.configure(lm=lm)

class ExtractInfo(dspy.Signature):
    """you are gonna be given an abtract of a patent and you need to generate 3 queries that
    can be used to search for prior art patents related to the given patent abstract. 
    The queries should be concise and relevant to the key aspects of the patent abstract.
    only return the queries as a json array of strings with no other text."""

    patent_abstract: str = dspy.InputField()
    number_of_queries: int = dspy.InputField(default=3, desc="Number of search queries to generate.")
    queries: list[str] = dspy.OutputField(desc="List of search queries for prior art patents.")

module = dspy.Predict(ExtractInfo)

def get_queries_from_abstract(patent_abstract: str, number_of_queries=3) -> list[str]:
    """Generate search queries for prior art patents based on the given patent abstract."""
    try:
        return module(patent_abstract=patent_abstract, 
                      number_of_queries=number_of_queries).queries
    except Exception as e:
        raise print(f"Error generating queries: {e}")
        
    
def create_search_queries(patent_data: list[str], number_of_queries=3)-> pd.DataFrame:
    """Generate search queries for a list of patent abstracts."""
    all_queries = []
    for patent in patent_data:
        queries = get_queries_from_abstract(patent['abstract'], number_of_queries)
        for query in queries:
            all_queries.append({'publication_number': patent['publication_number'], 'query': query, 'abstract': patent['abstract']})
    out = pd.DataFrame(all_queries)
    out.to_csv("patent_search_queries.csv", index=False)
    return out


In [39]:
df = create_search_queries(patent_data, number_of_queries=3)

2025/11/16 18:19:50 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Error generating queries: litellm.RateLimitError: litellm.RateLimitError: geminiException - {
  "error": {
    "code": 429,
    "message": "You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-2.0-flash\nPlease retry in 5.785135679s.",
    "status": "RESOURCE_EXHAUSTED",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.Help",
        "links": [
          {
            "description": "Learn more about Gemini API quotas",
            "url": "https://ai.google.dev/gemini-api/docs/rate-limits"
          }
        ]
      },
      {
        "@type": "type.googleapis.com/google.rpc.QuotaFailure",
        "violations": [
          {
            "quot

2025/11/16 18:20:24 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Error generating queries: litellm.RateLimitError: litellm.RateLimitError: geminiException - {
  "error": {
    "code": 429,
    "message": "You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-2.0-flash\nPlease retry in 32.322554883s.",
    "status": "RESOURCE_EXHAUSTED",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.Help",
        "links": [
          {
            "description": "Learn more about Gemini API quotas",
            "url": "https://ai.google.dev/gemini-api/docs/rate-limits"
          }
        ]
      },
      {
        "@type": "type.googleapis.com/google.rpc.QuotaFailure",
        "violations": [
          {
            "quo

2025/11/16 18:20:34 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Error generating queries: litellm.RateLimitError: litellm.RateLimitError: geminiException - {
  "error": {
    "code": 429,
    "message": "You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-2.0-flash\nPlease retry in 22.596334785s.",
    "status": "RESOURCE_EXHAUSTED",
    "details": [
      {
        "@type": "type.googleapis.com/google.rpc.Help",
        "links": [
          {
            "description": "Learn more about Gemini API quotas",
            "url": "https://ai.google.dev/gemini-api/docs/rate-limits"
          }
        ]
      },
      {
        "@type": "type.googleapis.com/google.rpc.QuotaFailure",
        "violations": [
          {
            "quo

2025/11/16 18:20:43 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


KeyboardInterrupt: 

### RULER:  LLM-as-a-judge
As in RULER from openpipe


RULER (Relative Universal LLM-Elicited Rewards) is a general-purpose reward function that uses an LLM-as-judge to rank multiple agent trajectories. 

It requires no labeled data, expert feedback, or hand-crafted reward functions, yet reliably improves agent performance.